# CVTD — Data Generation on Colab A100/H100 (Qwen3 + vLLM)

ToolACE-adapted **construct-then-paraphrase**: gold args built from closed catalogues (always valid);
model only writes the Vietnamese query. Generates ~5 500 train / ~2 000 eval at `SCALE=2.5`.

**GPU requirement:**

| Model | VRAM (fp16) | Fits on |
|-------|-------------|---------|
| `Qwen/Qwen3-32B-Instruct` | ~64 GB | A100 80 GB, H100 |
| `Qwen/Qwen3-14B-Instruct` | ~28 GB | A100 40 GB ✅ (standard Colab) |

Set `GEN_MODEL` in Cell 3 accordingly. vLLM loads in fp16 by default.

In [ ]:
# 1) Clone repo — real_tools.json, reference codes, station catalogue are git-tracked
REPO_URL = "https://github.com/<you>/telco_function_calling.git"  # <-- set your URL
import os
if not os.path.isdir('telco_function_calling'):
    !git clone $REPO_URL
%cd telco_function_calling

# Install vLLM + transformers (~2 min)
!pip install -q vllm transformers accelerate bitsandbytes
!nvidia-smi -L

In [ ]:
# 2) Configure — Qwen3-32B with 4-bit quantization fits A100 40 GB (~18 GB)
import os

os.environ['GEN_MODEL']     = 'Qwen/Qwen3-32B-Instruct'
os.environ['QUANTIZATION']  = 'bitsandbytes'  # 4-bit; unset for fp16 on A100 80GB / H100
os.environ['SCALE']         = '2.5'   # train-family multiplier → ~5 500 train
os.environ['EVAL_SCALE']    = '1.7'   # eval-family multiplier  → ~2 000 eval
os.environ['BACKEND']       = 'vllm'
os.environ['DATA_DIR']      = 'data'  # real_tools.json etc. are git-tracked in repo
os.environ['OUT_DIR']       = '/content/outputs'
os.makedirs('/content/outputs', exist_ok=True)

print(f"Model : {os.environ['GEN_MODEL']}  (4-bit bitsandbytes ~18 GB)")
print(f"Scale : train x{os.environ['SCALE']}  eval x{os.environ['EVAL_SCALE']}")

In [ ]:
# 3) Run generation  (~60-120 min depending on GPU + model)
!python kaggle/generate_real_data_vllm.py

In [ ]:
# 4) Inspect output counts + save to Drive
import glob, os

for f in sorted(glob.glob('/content/outputs/*.jsonl')):
    print(os.path.basename(f), sum(1 for _ in open(f)))
print()
print(open('/content/outputs/real_data_generation.json').read())

# Persist to Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cvtd/gen_outputs
!cp /content/outputs/*.jsonl /content/drive/MyDrive/cvtd/gen_outputs/
!cp /content/outputs/real_data_generation.json /content/drive/MyDrive/cvtd/gen_outputs/
print('Saved -> Drive: MyDrive/cvtd/gen_outputs/')